In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1,1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
plt.scatter(X[:,0], X[:,1], c=y, cmap='viridis')
plt.title("Dataset Visualization")
plt.show()

In [ ]:
class BadMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 1)
        
    def forward(self, x):
        x = torch.sigmoid(self.fc1(x))   # bad: sigmoid hidden
        x = torch.tanh(self.fc2(x))      # bad combo
        x = torch.relu(self.fc3(x))      # bad: relu output
        return x

In [ ]:
model = BadMLP()
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=1.0)  # large LR

In [ ]:
losses = []
grad_history = []

for epoch in range(50):
    optimizer.zero_grad()
    
    outputs = model(X_train)
    outputs = torch.clamp(outputs, 0, 1)
    
    loss = criterion(outputs, y_train)
    loss.backward()
    
    grad_norm = model.fc1.weight.grad.norm().item()
    grad_history.append(grad_norm)
    
    optimizer.step()
    
    losses.append(loss.item())

print("Final Loss:", losses[-1])

In [ ]:
plt.plot(losses)
plt.title("Loss Curve (Bad Model)")
plt.show()

In [ ]:
plt.plot(grad_history)
plt.title("Gradient Norm (fc1)")
plt.show()

In [ ]:
with torch.no_grad():
    h1 = torch.sigmoid(model.fc1(X_train))
    h2 = torch.tanh(model.fc2(h1))

plt.hist(h1.numpy().flatten(), bins=50)
plt.title("Hidden Layer 1 Activations")
plt.show()

plt.hist(h2.numpy().flatten(), bins=50)
plt.title("Hidden Layer 2 Activations")
plt.show()

In [ ]:
OBSERVATIONS (Write in exam)

Ans:

Loss does not decrease properly → training stagnation
Gradients either:
very small → vanishing gradient (sigmoid saturation)
unstable due to high LR → exploding updates
Activation histogram:
sigmoid outputs near 0 or 1 → saturation
Output layer ReLU:
negative logits become 0 → dead output behavior

DIAGNOSIS
Ans:

Sigmoid in hidden layer
→ saturation → vanishing gradients
Large learning rate (1.0)
→ unstable updates → exploding gradients
ReLU in output
→ invalid for binary classification
Tanh + bad initialization
→ inconsistent scaling

In [ ]:
class GoodMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))      # fix
        x = torch.relu(self.fc2(x))      # fix
        x = torch.sigmoid(self.fc3(x))   # fix
        return x

In [ ]:
model = GoodMLP()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
losses_fixed = []

for epoch in range(50):
    optimizer.zero_grad()
    
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    loss.backward()
    optimizer.step()
    
    losses_fixed.append(loss.item())

print("Final Loss:", losses_fixed[-1])

In [ ]:
plt.plot(losses, label="Bad")
plt.plot(losses_fixed, label="Fixed")
plt.legend()
plt.title("Loss Comparison")
plt.show()

In [ ]:
# FINAL JUSTIFICATION 

Ans:

Replaced sigmoid with ReLU in hidden layers
→ avoids saturation → better gradient flow
Replaced ReLU output with sigmoid
→ correct for binary classification
Reduced learning rate
→ stabilizes training
Used Adam optimizer
→ adaptive updates → faster convergence

Result:

Stable gradients
Proper learning
Loss decreases smoothly
No dead neurons or stagnation